# Capstone #1: Research Agent

*Notebook 11*

Put it all together.

Combine file search and Code Interpreter into one agent.

It retrieves information from reference documents, runs the numbers, and produces a structured report.


---

## 🔧 Setup

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from openai import OpenAI
from agents import Agent, CodeInterpreterTool, FileSearchTool, Runner, ToolCallItem

MODEL = "gpt-5-mini"
REASONING_MODEL = "gpt-5"

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def called_tool_types(result) -> set[str]:
    """The set of hosted tool-call types the run actually invoked."""
    return {
        getattr(item.raw_item, "type", "")
        for item in result.new_items
        if isinstance(item, ToolCallItem)
    }


print(f"✅ Setup complete: Using {MODEL}")

---

## 🏗️ System Architecture

The research agent combines two built-in tools from Lessons 09-10:

- **File Search** (Lesson 09): retrieves facts from reference documents

- **Code Interpreter** (Lesson 10): computes and compares quantitative data

**Workflow:** Topic → agent retrieves from the documents and runs the numbers → structured report.

---

## 📄 Phase 1: Build the Knowledge Base

We'll create two reference documents and upload them to a vector store.

The agent researches from both: an internal survey and a synthetic industry benchmark.

In [ ]:
# Two reference documents (both SYNTHETIC demo data, course fixtures): an internal survey and a cross-industry benchmark.

# Document 1: internal survey
survey_doc = """# AI Adoption Survey (Synthetic Demo Data): Internal Research Summary

## Survey Overview
Fictional survey created for this course: 847 respondents across technology, healthcare,
finance, and retail sectors. Companies ranged from 50 to 10,000+ employees.

## Key Findings

### Adoption Rates
- 73% of companies surveyed are actively using AI tools in production
- 18% are in pilot or evaluation phase
- 9% have no AI initiatives currently

### Primary Use Cases (ranked by adoption)
1. Customer support automation: 61%
2. Data analysis and reporting: 58%
3. Content generation: 47%
4. Code assistance: 44%
5. Document processing: 39%

### Biggest Barriers to Adoption
- Data privacy and security concerns: 67%
- Integration with existing systems: 54%
- Lack of skilled staff: 48%
- Cost and ROI uncertainty: 43%

### Budget Trends
- Average AI budget increase year-over-year: 34%
- Companies with dedicated AI teams: 41%
- Expected to increase AI investment next year: 78%

### Satisfaction
- Exceeded expectations: 29%
- Met expectations: 51%
- Below expectations: 20%
"""

# Document 2: industry benchmark (synthetic figures, course-owned fixture)
benchmark_doc = """# Cross-Industry AI Adoption Benchmark (Synthetic Demo Data)

Illustrative course fixture. Figures are synthetic - do not cite as real.
Source: Course fixture, ai_adoption_benchmark.txt

## Adoption
- 68% of organizations report AI in production
- 22% in pilot or evaluation
- 10% no AI initiatives

## Barriers (ranked)
- Data privacy and security: 64%
- Integration with existing systems: 57%
- Talent and skills gaps: 45%
- Cost and ROI uncertainty: 46%

## Investment
- Median AI budget increase year-over-year: 29%
- Organizations planning increased investment next year: 71%
"""

# Save locally
survey_path = Path("ai_adoption_survey.txt")
survey_path.write_text(survey_doc)
benchmark_path = Path("ai_adoption_benchmark.txt")
benchmark_path.write_text(benchmark_doc)
doc_paths = [survey_path, benchmark_path]

# --------------------------------------------------------------
print(f"✅ Reference documents created: {', '.join(p.name for p in doc_paths)}")

#### Upload to Vector Store

In [ ]:
print("Uploading to vector store...\n")

vector_store = client.vector_stores.create(name="Research Agent Knowledge Base")

with open(survey_path, "rb") as survey_file, open(benchmark_path, "rb") as benchmark_file:
    file_batch = client.vector_stores.file_batches.upload_and_poll(
        vector_store_id=vector_store.id,
        files=[survey_file, benchmark_file]
    )

if file_batch.file_counts.failed:
    raise RuntimeError(f"File processing failed: {file_batch.file_counts.failed} file(s)")
if file_batch.file_counts.completed != len(doc_paths):
    raise RuntimeError(
        f"Expected {len(doc_paths)} files processed, got {file_batch.file_counts.completed}"
    )

# --------------------------------------------------------------
print(f"✅ Vector store ready: {vector_store.id}")
print(f"   Files: {file_batch.file_counts.completed} processed")

With the knowledge base ready, we can build one agent.

It answers from both reference documents: the internal survey and the industry benchmark.

## 🤖 Phase 2: Build the Research Agent

Both tools live in one agent, and its instructions tell it how to use each.

Pattern to copy:

- List every tool the agent might need

- Give numbered instructions for when to use each

- Start with default parameters

In your own project, swap the vector store contents, instructions, and `ResearchReport` fields.

The report format is deliberately constrained: one-sentence findings, summaries instead of pasted text, and citations only in the source list.

Hosted-tool output remains model-generated. Validate important source facts and citation metadata before relying on the report.

In [ ]:
from pydantic import BaseModel
from typing import List

class ResearchReport(BaseModel):
    executive_summary: str
    key_findings: List[str]
    data_analysis: str
    sources: List[str]

research_instructions = (
    "You are a thorough research analyst. When given a research topic:\n\n"
    "1. Use file search to retrieve the relevant facts from BOTH reference documents\n"
    "2. Use the Python tool to compute and compare the numbers\n"
    "Report only the final values computed with Python: do not reproduce formulas or intermediate arithmetic.\n"
    "Both documents are synthetic demo data: say so in the report.\n"
    "Preserve source numbers exactly: never merge, alter, or invent them.\n"
    "Keep the executive summary to 2-3 sentences and each key finding to ONE sentence.\n"
    "Summarize documents in your own words: never paste document text verbatim.\n"
    "In Sources, list the exact filenames you used: ai_adoption_survey.txt and ai_adoption_benchmark.txt.\n"
    "3. Synthesize findings into a structured report with these sections:\n"
    "    - Executive Summary (2-3 sentences)\n"
    "    - Key Findings (bullet points)\n"
    "    - Data Analysis (numbers and statistics)\n"
    "    - Sources\n\n"
    "Be specific with numbers."
)

research_agent = Agent(
    name="ResearchAgent",
    instructions=research_instructions,
    model=MODEL,
    output_type=ResearchReport,
    tools=[
        FileSearchTool(
            vector_store_ids=[vector_store.id],
            max_num_results=5
        ),
        CodeInterpreterTool(tool_config={
            "type": "code_interpreter",
            # Auto mode creates a container or reuses one already in model context.
            "container": {"type": "auto"}
        })
    ]
)

# --------------------------------------------------------------
print("✅ Research agent ready")
print("    Tools: FileSearch + CodeInterpreter")

---

## 🚀 Phase 3: Run the Research Agent

The agent can use file search and Code Interpreter in one run.

⚠️ **Security note:** Tool output (retrieved doc chunks) is untrusted.

Validate before passing it downstream (more in Lesson 21).

In [ ]:
required_tools_ok = False  # a failed OR tool-missing run does not demonstrate the objective

research_topic = (
    "Using file search, retrieve our internal survey's AI-adoption rate, respondent count, "
    "and leading barrier, and the industry benchmark's adoption rate and leading barrier. "
    "Then use the Python tool to compute the number of survey respondents using AI in production "
    "(nearest whole respondent) and the percentage-point difference between our adoption rate and "
    "the benchmark's. Compare the leading barriers across the two documents. "
    "List the exact source filenames in Sources."
)

print("🔬 RESEARCH AGENT RUNNING")
print("=" * 60)
print(f"Topic: {research_topic}")
print("\nResearching...\n")

# Multi-tool runs can fail at any step, so surface failures clearly
try:
    research_result = await Runner.run(research_agent, input=research_topic)
except Exception as error:
    print(f"Research failed: {error}")
    research_result = None

if research_result:
    report = research_result.final_output

    print("=" * 60)
    print("📋 RESEARCH REPORT")
    print("=" * 60)
    print(f"\nExecutive Summary:\n{report.executive_summary}\n")
    print("Key Findings:")
    for finding in report.key_findings:
        print(f"  • {finding}")
    print(f"\nData Analysis:\n{report.data_analysis}\n")
    print("Sources:")
    for source in report.sources:
        print(f"  • {source}")
    print("=" * 60)

    # Because we used output_type=ResearchReport, `report` is a validated Python object
    # with the expected shape for downstream code. Facts and sources still need review.

    # Evidence: both tools are required for this capstone
    called = called_tool_types(research_result)
    expected_tools = {
        "File Search": "file_search_call",
        "Code Interpreter": "code_interpreter_call",
    }
    missing = [label for label, ct in expected_tools.items() if ct not in called]
    print("\nTool use:")
    for label, ct in expected_tools.items():
        print(f"  {label}: {'yes' if ct in called else 'no'}")
    if missing:
        print(f"\n❌ Objective NOT demonstrated - required tools did not run: {', '.join(missing)}")
    required_tools_ok = not missing

    # Sanity checks (not proof of correctness): the fixture facts should survive
    report_text = " ".join(
        [report.executive_summary, report.data_analysis] + report.key_findings
    ).lower()
    sources_text = " ".join(report.sources).lower()
    diff_ok = any(phrase in report_text for phrase in
                  ["5 percentage points", "five percentage points", "5-point gap", "five-point gap"])
    print("\nSanity checks (not proof of correctness):")
    for label, ok in {
        "synthetic-data disclosure present": "synthetic" in report_text,
        "survey 847 present": "847" in report_text,
        "survey 73% present": "73%" in report_text,
        "benchmark 68% present": "68%" in report_text,
        "618 estimate present": "618" in report_text,
        "5 percentage-point gap": diff_ok,
        "survey file cited": "ai_adoption_survey.txt" in sources_text,
        "benchmark file cited": "ai_adoption_benchmark.txt" in sources_text,
    }.items():
        print(f"  {label}: {'✅' if ok else '❌'}")
else:
    print("\n❌ Run failed: no report produced.")

## 🧹 Demo Cleanup

Removes the demo's sample files so the exercises start clean.

In [ ]:
# Clean up local sample files before exercises
for file in ["ai_adoption_survey.txt", "ai_adoption_benchmark.txt"]:
    path = Path(file)
    if path.exists():
        path.unlink()
        print(f"✅ Local file deleted: {file}")

---

## 💪 Practice Exercises

### Exercise 1: Export Research Report to Markdown

*Covers: saving agent output, writing results to a file*

Save the research report to a `.md` file using the existing `research_result`.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Export Research Report to Markdown
# --------------------------------------------------------------
# Objective: Save the existing research report to a markdown file.

from datetime import datetime

# TODO 1: Create a markdown header with the topic and current date
#          Hint: date_str = datetime.now().strftime("%Y-%m-%d")
#                header = f"# Research Report: {research_topic}\nDate: {date_str}\n\n"

# TODO 2: Build the report body from all four ResearchReport fields:
#          executive_summary, key_findings, data_analysis, sources
#          (research_result.final_output.executive_summary, and so on)

# TODO 3: Write the combined content to "report.md" using Path("report.md").write_text()

# TODO 4: Print a confirmation message showing the file path

# TODO 5: (Optional) Delete report.md when you're done to keep the folder clean

# --- Write your code below this line ---

---

### Exercise 2: Evaluate One Research Report

*Covers: judge-agent evaluation, scoring a structured report*

Apply the judge-agent evaluation pattern from Lesson 07 to one research report.

Focus on the judge step, not a full test harness.

Good golden tests for this kind of agent check two things:

- **Content coverage**: did it include required facts?

- **Tool-backed grounding**: did it cite sources or use internal data when expected?

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 2: Evaluate One Research Report
# --------------------------------------------------------------
# Objective: Score one research report using a judge agent (Lesson 07 pattern).

from pydantic import BaseModel, Field

# A golden test: the topic we already ran, with criteria the report should meet
golden_test = {
    "topic": "Current state of AI adoption in enterprise",
    "must_contain": ["73%", "68%", "618", "privacy"],  # facts from both docs + the calculation
}

# Define an EvalResult model for the judge agent
# (no passed field: pass/fail comes from the deterministic checks below,
#  the judge only scores quality)
class EvalResult(BaseModel):
    score: int = Field(ge=0, le=10)
    feedback: str

# TODO 1: Check the golden_test criteria deterministically in Python:
#          confirm each must_contain string appears in
#          report.model_dump_json()

# TODO 2: Create a judge agent using REASONING_MODEL and output_type=EvalResult
#          Instructions: score the report's clarity, grounding, and usefulness

# TODO 3: Build the judge input from golden_test and
#          report.model_dump_json(). Judge the captured report.
#          Do not rerun the research agent.

# TODO 4: Check tool use deterministically:
#          required = {"file_search_call", "code_interpreter_call"}
#          Confirm required <= called_tool_types(research_result)

# TODO 5: Derive the final verdict in Python:
#          all deterministic checks pass AND the judge score >= 7

# --- Write your code below this line ---

---

## 🎯 Key Takeaways

**One agent can choose among its tools:**

- File search to retrieve facts from reference documents

- Code Interpreter for computation and comparison

- Inspect run items to prove which tools actually ran
<br>
<br>

**Instructions guide tool use:**

- Explicit instructions about when to use each tool produce more consistent results

- Structured output (`output_type=ResearchReport`) guarantees the report's shape on a successful run, not its content

- Handle errors where the caller can recover: a multi-tool run can fail at any step
<br>
<br>

**Files and vector stores are separate resources:**

- Delete uploaded files with `client.files.delete(...)` and the store with `client.vector_stores.delete(...)`

- In a real app, keep the store and reuse its ID until the documents change

---

## 📍 Next Step

**Notebook 12: Handoffs**  

Learn how to route tasks between specialized agents using the handoff pattern.

---

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-11-capstone-1--research-agent)

---

#### Cleanup: Delete Uploaded Files and Vector Store

Uploaded files and vector stores persist on OpenAI's servers until deleted.

Deleting the store does not delete the files you uploaded, so remove both.

Run this cell when you're done to avoid ongoing storage charges.

In [ ]:
# Vector stores and uploaded Files are separate objects: delete both
uploaded_file_ids = [
    store_file.id
    for store_file in client.vector_stores.files.list(
        vector_store_id=vector_store.id
    ).data
]

for file_id in uploaded_file_ids:
    client.files.delete(file_id)

client.vector_stores.delete(vector_store.id)

print(f"✅ Uploaded files deleted: {len(uploaded_file_ids)}")
print(f"✅ Vector store deleted: {vector_store.id}")

# Enforce the capstone objective AFTER cleanup so nothing leaks on failure
if not required_tools_ok:
    raise RuntimeError(
        "Capstone did not demonstrate its objective (required tool missing or run failed) - see the run cell above."
    )

---